In [ ]:
import pandas as pd

# === Step 1: Load the data files ===
file_inventory = "aliquot_DCC_data_release_v2-0-5_inventory.tsv"
file_shipped = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
file_obs = "follow-up_obs_DCC_data_release_v2-0-1.tsv"
file_rct = "follow-up_rct_DCC_data_release_v2-0-1.tsv"

df_inventory = pd.read_csv(file_inventory, sep="\t", dtype=str)
df_shipped = pd.read_csv(file_shipped, sep="\t", dtype=str)
df_obs = pd.read_csv(file_obs, sep="\t", dtype=str)
df_rct = pd.read_csv(file_rct, sep="\t", dtype=str)

# === Step 2: Extract unique submitter IDs ===
followup_ids_inventory = df_inventory["*follow_ups.submitter_id"].dropna().unique()
followup_ids_shipped = df_shipped["*follow_ups.submitter_id"].dropna().unique()
all_followup_ids = set(followup_ids_inventory).union(followup_ids_shipped)

submitter_ids_obs = df_obs["*submitter_id"].dropna().unique()
submitter_ids_rct = df_rct["*submitter_id"].dropna().unique()
all_submitter_ids = set(submitter_ids_obs).union(submitter_ids_rct)

# === Step 3: Find missing follow-up submitter IDs ===
missing_followups = sorted(all_followup_ids - all_submitter_ids)

# === Step 4: Define output header ===
columns = [
    "*type", "project_id", "*submitter_id", "cases.submitter_id", "demographics.submitter_id",
    "*days_to_follow_up", "admission_date", "ascites_culture", "ascites_culture_result",
    "ascites_date", "ascites_diagnosis_date", "ascites_organism", "bariatric_surgery",
    "blood_culture", "blood_culture_date", "blood_culture_result", "blood_organism", "bmi",
    "cdc_hiv_risk_factors", "child_pugh_score", "ct_date", "ct_result", "ct_result_finding",
    "ct_result_sig", "duodenum_ulcer_bleed", "duodenum_ulcer_size", "endoscopy", "endoscopy_date",
    "endoscopy_ulcer_present", "endoscopy_varices_present", "esophageal_ulcer_bleed",
    "esophageal_ulcer_size", "esophageal_varices_bleed", "esophageal_varices_size", "event_type",
    "fibro_capmed", "fibro_e", "fibro_iqr", "fibro_probe", "fibro_total_number", "fibroscan",
    "fibroscan_date", "firbro_e_iqr", "firbro_valid_number", "gastric_ulcer_bleed",
    "gastric_ulcer_size", "gastric_varices_bleed", "gastric_varices_size", "height",
    "hep_carcinoma", "hep_enceph", "hep_enceph_diagnosis_date", "hepcar_diagnosis_date",
    "hospitalized_alc_hep", "hospitalized_alc_hep_times", "hospitalized_at_enrollment",
    "infection_screen_date", "infection_screen_done", "lille_score", "liver_abdomen_imaging",
    "liver_imaging_type", "liver_score_collected", "liver_score_date", "liver_transplant",
    "liver_transplant_date", "maddreys_score", "medical_info_collected", "meld_score", "mri_date",
    "mri_finding", "mri_result", "mri_sig", "other_imaging_date", "other_imaging_finding",
    "other_imaging_result", "other_imaging_sig", "other_imaging_type",
    "portal_hypertensive_gastropathy", "soc_collected", "stool_test", "stool_test_date",
    "stool_test_finding", "tlfb_collected", "tlfb_drinking_days", "tlfb_number_drinks",
    "ultrasound_date", "ultrasound_finding", "ultrasound_result", "ultrasound_result_sig",
    "urine_culture", "urine_culture_date", "urine_culture_fungal_result", "urine_culture_organism",
    "urine_culture_result", "varices", "varices_diagnosis_date", "visit_day", "weight", "xray",
    "xray_date", "xray_findings", "xray_impression", "xray_infiltrates", "xray_last_4_weeks",
    "xray_normal", "xray_pleural_effusion"
]

# === Step 5: Build output rows ===
rows = []
for submitter_id in missing_followups:
    parts = submitter_id.split("_")
    if len(parts) >= 3:
        case_id = f"{parts[0]}_{parts[1]}"
        days = parts[2]
    else:
        case_id = parts[0]
        days = ""

    row = {
        "*type": "follow_up",
        "project_id": "ARDaC-AlcHepNet",
        "*submitter_id": submitter_id,
        "cases.submitter_id": case_id,
        "demographics.submitter_id": f"{case_id}_demographic",
        "*days_to_follow_up": days
    }

    # Fill remaining columns with empty strings
    for col in columns:
        if col not in row:
            row[col] = ""

    rows.append(row)

# === Step 6: Create DataFrame and export ===
df_output = pd.DataFrame(rows, columns=columns)
df_output.to_csv("follow-up_add-on_DCC_data_release_v2-0-8.tsv", sep="\t", index=False)
print("File saved as follow-up_add-on_DCC_data_release_v2-0-8.tsv")